# Lezione 58 — La pipeline: MemoryAILab

> **Modulo:** capstone · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezioni 52-57.
>
> Obiettivo pratico unico: assemblare tutti i componenti in un'unica classe
> **`MemoryAILab`** che, data una memoria testuale, produce il **record
> strutturato completo** definito nella Lezione 52 — l'obiettivo del corso.

## Teoria essenziale

Ora sostituiamo gli stub della Lezione 52 con i componenti reali (Lezioni 54-56):
classificazione, rappresentazione, relazioni, importanza. `MemoryAILab.process`
li orchestra e restituisce il record. La regola `should_store` applica una soglia
all'importanza: il lab decide **cosa vale la pena ricordare**.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

proc = Path('..') / 'datasets' / 'processed'
TIPI = ["episodic", "semantic", "preference"]
_idx = {t: i for i, t in enumerate(TIPI)}
_train = pd.read_csv(proc / 'memory_train.csv')
_train = _train[_train['type'].isin(TIPI)].reset_index(drop=True)

def _tok(t):
    return re.findall(r"[a-zA-Z]+", str(t).lower())

# --- classificatore bag-of-words + softmax (Lezione 54) ---
_vocab = {}
for _t in _train['text']:
    for _w in _tok(_t):
        _vocab.setdefault(_w, len(_vocab))

def _bow(testo):
    v = np.zeros(len(_vocab))
    for w in _tok(testo):
        if w in _vocab:
            v[_vocab[w]] += 1.0
    return v

def _softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

_Xtr = np.vstack([_bow(t) for t in _train['text']])
_ytr = np.array([_idx[t] for t in _train['type']])
_W = np.zeros((len(_vocab), len(TIPI)))
_b = np.zeros(len(TIPI))
_Y = np.eye(len(TIPI))[_ytr]
for _ in range(300):
    _P = _softmax(_Xtr @ _W + _b)
    _W -= 0.5 * (_Xtr.T @ (_P - _Y) / len(_Xtr) + 1e-3 * _W)
    _b -= 0.5 * (_P - _Y).mean(axis=0)

def classifica_tipo(testo):
    return TIPI[int(_softmax(_bow(testo)[None, :] @ _W + _b).argmax())]

# --- rappresentazione: embedding, entita' (Lezione 55) ---
DIM = 48
def embed(testo):
    v = np.zeros(DIM)
    for w in _tok(testo):
        v[int.from_bytes(w.encode(), 'little') % DIM] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def estrai_entita(testo):
    return [w for w in re.findall(r"\b[A-Z][a-zA-Z]+\b", str(testo)) if w not in {"The", "A", "User"}]

# --- relazioni: fallback a regole (Lezione 56) ---
_VERBI = {"visited": "visited", "met": "met", "likes": "likes",
          "prefers": "prefers", "works": "works_at", "lives": "lives_in"}
def estrai_relazioni(testo):
    ent = estrai_entita(testo)
    for v_sup, v_norm in _VERBI.items():
        if v_sup in testo.lower() and len(ent) >= 2:
            return [{"source": ent[0], "type": v_norm, "target": ent[1]}]
    return []

# --- importanza composita (Lezione 25, versione compatta) ---
_FORTI = {"prefers", "likes", "dislikes", "hates", "loves", "always", "never", "important"}
def calcola_importanza(testo):
    # composita (Lezione 25): lunghezza + parole forti + numero di entita'
    parole = _tok(testo)
    forti = sum(1 for w in parole if w in _FORTI)
    lung = min(len(parole) / 15.0, 1.0)
    n_ent = len(estrai_entita(testo))
    imp = 0.30 * lung + 0.40 * min(forti, 1) + 0.20 * min(n_ent, 2) / 2 + 0.10
    return round(float(min(imp, 1.0)), 2)

In [2]:
import json

class MemoryAILab:
    def __init__(self, soglia_store=0.4):
        self.soglia = soglia_store

    def process(self, memory_id, testo, timestamp=None):
        tipo = classifica_tipo(testo)
        entita = estrai_entita(testo)
        imp = calcola_importanza(testo)
        return {
            "memory_id": memory_id,
            "text": testo,
            "timestamp": timestamp,
            "type": tipo,
            "entities": entita,
            "importance": imp,
            "should_store": imp >= self.soglia,
            "embedding_dim": DIM,
            "relations": estrai_relazioni(testo),
        }

lab = MemoryAILab()
record = lab.process("mem_001", "Marco visited Glasgow with his son.",
                     timestamp="2026-07-03")
print(json.dumps(record, indent=2, ensure_ascii=False))

{
  "memory_id": "mem_001",
  "text": "Marco visited Glasgow with his son.",
  "timestamp": "2026-07-03",
  "type": "episodic",
  "entities": [
    "Marco",
    "Glasgow"
  ],
  "importance": 0.42,
  "should_store": true,
  "embedding_dim": 48,
  "relations": [
    {
      "source": "Marco",
      "type": "visited",
      "target": "Glasgow"
    }
  ]
}


Leggi il record: e' esattamente il contratto della Lezione 52 (e dello schema
obiettivo nel `COURSE_FACTORY_SPEC.md`), popolato da componenti reali. Un solo
`process` produce tipo, entita', relazioni, importanza e la decisione di
memorizzazione.

In [3]:
# controlli di non-regressione sull'intera pipeline
assert record["type"] in TIPI
assert record["entities"] == ["Marco", "Glasgow"]
assert record["relations"] == [{"source": "Marco", "type": "visited", "target": "Glasgow"}]
assert 0.0 <= record["importance"] <= 1.0
assert isinstance(record["should_store"], bool)

# tutti i campi del contratto presenti
attesi = {"memory_id", "text", "timestamp", "type", "entities",
          "importance", "should_store", "embedding_dim", "relations"}
assert set(record.keys()) == attesi
print("pipeline completa: record conforme al contratto  OK")

pipeline completa: record conforme al contratto  OK


## Riepilogo (max 8 punti)

1. `MemoryAILab.process` orchestra tutti i componenti reali.
2. Sostituisce gli stub della Lezione 52, struttura invariata.
3. Produce il **record strutturato completo** del contratto.
4. Classificazione, entita', relazioni, importanza in un colpo.
5. `should_store` = soglia sull'importanza: cosa ricordare.
6. Il record coincide con lo schema obiettivo della spec.
7. Un `assert` verifica la conformita' al contratto.
8. E' l'obiettivo del corso, funzionante end-to-end.

## Quiz

1. Cosa decide la regola `should_store`?
2. Perche' la struttura della pipeline non e' cambiata dalla Lezione 52?
3. Quali Lezioni forniscono i componenti reali?

*(Risposte: 1. se una memoria supera la soglia di importanza e va memorizzata;
2. perche' i componenti avevano firme stabili: abbiamo solo sostituito gli stub;
3. 54 (tipo), 55 (embedding/entita'), 56 (relazioni), 25 (importanza).)*